# Fine-Tuning Large Language Models

## 1. Import Required Dependencies & Variables

In [1]:
import os
import gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
import pandas as pd
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from peft import PeftModel, LoraConfig
from timeit import default_timer as timer
from huggingface_hub import login, snapshot_download
from peft import get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

from dotenv import load_dotenv
load_dotenv()

/mnt/c/Users/konda/Desktop/fine-tuning-llms/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("❌ Missing HF_TOKEN in .env file!")

HF_REPO_ID = "google/gemma-3-1b-it"
LOCAL_MODEL_PATH = "models/models--google--gemma-3-1b-it"

LORA_MODE = "qlora" # or "qlora"
ADAPTER_DIR = "./sft_output/adapters"
DATASET_PATH = '/mnt/c/Users/konda/Desktop/Sentiment Analysis for Mental Health Dataset.csv'

## 2. Download the LLM Model from Hugging Face hub



In [3]:
def download_model(hf_token: str = HF_TOKEN, hf_repo_id: str = HF_REPO_ID):
    """Authenticate and download a HF model."""
    
    login(token=hf_token)

    local_dir = snapshot_download(
        repo_id = hf_repo_id,   # Hugging Face model repository
        revision = "main",      # Branch, tag, or commit to download
        cache_dir = "./models"  # Local directory to store the downloaded model
    )

    print(f"Model downloaded to: {local_dir}")
    return local_dir

download_model()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 6791.30it/s]

Model downloaded to: ./models/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752


'./models/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752'

## 2. Set the Trainable Adapters (LoRa & QLoRA)

Trainable parameters reported by LoRA directly correspond to the part of the model that can learn and adapt during fine-tuning. We optimize only this subset of the model, leaving the rest frozen.

In [4]:
# LoRA adapter configuration
lora_cfg = LoraConfig(
    task_type="CAUSAL_LM",                   # Causal language modeling task
    
    target_modules = [                       # Layers where LoRA adapters are injected
        "q_proj", "k_proj", "v_proj",        # Attention projections: query, key, and value
        "o_proj",                            # Attention output projection
        "gate_proj", "up_proj", "down_proj"  # MLP gating + up / down projections
    ],

    r=8,                         # Rank: size of the low-rank matrices (↑ r ⇒ ↑ capacity & VRAM usage)
    lora_alpha=16,               # Scaling factor for LoRA updates (often >= r; controls update magnitude)
    lora_dropout=0.05,           # Dropout on LoRA layers 
    
    bias="none",                 # Do not add/train separate LoRA bias parameters; keep original biases as-is
    use_rslora=False,            # Disable Rank-Stabilised LoRA (enable if you need extra stability at higher r)
)


# QLoRA adapter configuration
qlora_cfg = LoraConfig(
    task_type="CAUSAL_LM",                   # Causal language modeling task
    
    target_modules = [                       # Layers where LoRA adapters are injected
        "q_proj", "k_proj", "v_proj",        # Attention projections: query, key, and value
        "o_proj",                            # Attention output projection
        "gate_proj", "up_proj", "down_proj"  # MLP gating + up / down projections
    ],

    r=64,                        # LoRA rank — higher is normal in QLoRA (32–128 typical)
    lora_alpha=128,              # Scaling factor (2x r)
    lora_dropout=0.1,            # Dropout improves generalization for QLoRA

    bias="none",                 # Do not add/train separate LoRA bias parameters; keep original biases as-is
    use_rslora=False,            # Disable Rank-Stabilised LoRA (enable if you need extra stability at higher r)
)

## 3. Initialize the LLM Model

In [5]:
def load_model(model_id: str = HF_REPO_ID, inference: bool = False):
    """
    - If training: Loads base model, prepares k-bit (if qlora), and initializes new adapters.
    - If inference: Loads base model and attaches existing saved adapters.
    """
    
    # 1. Configure Quantization (Only for QLoRA)
    if LORA_MODE == "qlora":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,            
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        bnb_config = None
    
    # 2. Load Base Model
    print(f"⏳ Loading Base Model (Inference={inference})...")
    base_model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path = model_id,
        quantization_config=bnb_config,
        attn_implementation="eager",
        dtype="auto",
        device_map="auto" 
    )

    # Inference
    if inference:
        print(f"🔗 Loading existing adapters from {ADAPTER_DIR}...")
        model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
        model.eval() 
        print("✅ Model loaded successfully (Inference Mode).")
        return model
    
    # Training
    else:
        print("⚙️ Initializing new adapters for training...")
        
        # 3. Prepare Model for Training
        if LORA_MODE == "qlora":
            base_model = prepare_model_for_kbit_training(base_model)
            selected_config = qlora_cfg
        elif LORA_MODE == "lora":
            base_model.enable_input_require_grads()
            selected_config = lora_cfg
        else:
            raise ValueError(f"Invalid LORA_MODE: {LORA_MODE}")

        # 4. Initialize Adapter Structure
        model = get_peft_model(base_model, selected_config)
        model.print_trainable_parameters()
        
        print("✅ Model prepared successfully (Training Mode).")
        return model
    

In [6]:
model = load_model(model_id=HF_REPO_ID, inference=False)

⏳ Loading Base Model (Inference=False)...
⚙️ Initializing new adapters for training...
trainable params: 52,183,040 || all params: 1,052,068,992 || trainable%: 4.9600
✅ Model prepared successfully (Training Mode).


## 4. Initialize the Tokenizer 
Breaks down input text into smaller units(tokens) and assigns a unique integer ID to each token based on its vocabulary

In [ ]:
def load_tokenizer(model_id: str = HF_REPO_ID, inference: bool = False):
    """Loads and configures the tokenizer."""
    
    # 1. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # 2. Configure Padding Side
    if inference:
        # Inference (Generation): Padding on LEFT 
        # (So the model sees the prompt at the end and generates immediately after)
        tokenizer.padding_side = "left"
    else:
        # Training requires RIGHT padding so the model learns to predict the next token,
        # not the empty space.
        tokenizer.padding_side = "right"

    # 3. Fix Missing Pad Token
    # For Gemma/Llama to prevent crashes during batching.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    return tokenizer


def inspect_tokenizer(tokenizer: AutoTokenizer):
    """Prints details about the tokenizer to verify it works as expected."""
    # Test Tokenization
    text = "What is AI?"
    tokens = tokenizer.tokenize(text)
    print(f"Input Text: '{text}'")
    print(f"Tokens: {tokens}") 
    # Note: Gemma/Llama use specific symbols (like _) for spaces.
    # ['What', ' is', ' AI', '?']
    
    # Test Encoding (Text -> Numbers)
    # Each subword (or token) has a unique number (token ID) specific to the model's tokenizer and vocabulary, not universal across models.
    # A bos(begin of sequence) token will be add on the sequence to ensure that the model know where the sequence starts
    input_ids = tokenizer.encode(text, add_special_tokens=True)
    print(f"Token IDs: {input_ids}")
     # for instance [2, 3689, 563, 12498, 236881]

    # Test Decoding (Numbers -> Text)
    # This proves the tokenizer can reverse the process
    decoded_text = tokenizer.decode(input_ids)
    print(f"Decoded: '{decoded_text}'")
    
    print(f"BOS Token ID: {tokenizer.bos_token_id} ({tokenizer.bos_token})")
    print(f"EOS Token ID: {tokenizer.eos_token_id} ({tokenizer.eos_token})")
    print(f"PAD Token ID: {tokenizer.pad_token_id} ({tokenizer.pad_token})")
    print(f"Padding Side: {tokenizer.padding_side}") 
    # This is not the limit of what the tokenizer can process (it can tokenize infinite text).
    # This is the limit of the Model's Context Window (e.g., 4096 or 8192 tokens).
    # It acts as a strict instruction: "Truncate any input longer than this number."
    # If this prints a huge number (1e30), it means no limit is set.
    print(f"Model Max Length: {tokenizer.model_max_length}")

In [9]:
tokenizer = load_tokenizer(model_id=HF_REPO_ID, inference=False)
inspect_tokenizer(tokenizer=tokenizer)

Input Text: 'What is AI?'
Tokens: ['What', '▁is', '▁AI', '?']
Token IDs: [2, 3689, 563, 12498, 236881]
Decoded: '<bos>What is AI?'
BOS Token ID: 2 (<bos>)
EOS Token ID: 1 (<eos>)
PAD Token ID: 0 (<pad>)
Padding Side: right
Model Max Length: 1000000000000000019884624838656


After tokenizing, the model converts token IDs into embeddings through the embeddings layer. These embeddings capture semantic and contextual information about the tokens. Embeddings in models like transformers function similarly to how CNNs capture features, but instead of visual features (edges, textures, etc.), embeddings capture linguistic and semantic features of tokens. Example embeddings for each token (as vectors):

```python
Embeddings: [
    [0.1, 0.2, 0.3],  # Token 3689
    [0.5, 0.6, 0.7],  # Token 563
    [0.9, 0.8, 0.1],  # Token 12498
    [0.3, 0.4, 0.2]   # Token 236881
]

## 5. Data Preparation

In [10]:
# Normalize the Text
def normalize_text(text):
    """Sanitizes input text by removing newlines and tabs."""
    if not isinstance(text, str):
        return ""
    text = text.replace("\n", " ").replace("\t"," ")
    return text


def format_chat_template(row, tokenizer: AutoTokenizer):
    """Converts a data row into a structured chat format (User/Assistant), 
    then applies the tokenizer's chat template to generate a single training string."""
    
    chat = [
        {"role": "user", "content": row["statement"]},
        {"role": "assistant", "content": row["status"]}
    ]
    
    text = tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": text}


def prepare_dataset(dataset_path: str, tokenizer: AutoTokenizer):
    """Load CSV data, clean text, split into train/test sets (80/20), 
    and format inputs for the SFTTrainer."""
    
    # 1. Load and preprocess text
    df = pd.read_csv(dataset_path)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    df.dropna(subset=["statement", "status"], inplace=True)
    df["statement"] = df["statement"].apply(normalize_text)
    df["status"] = df["status"].apply(normalize_text)
    
    # 2. Convert to HF Dataset & Split
    full_dataset = Dataset.from_pandas(df, preserve_index=False)
    dataset_dict = full_dataset.train_test_split(test_size=0.2, seed=42)
    
    # 3. Apply Chat Template
    dataset_dict = dataset_dict.map(
            format_chat_template,
            fn_kwargs={"tokenizer": tokenizer}
        )

    # 4. Prepare Splits
    # Remove raw columns to save memory & prevent warnings in Trainer
    train_data = dataset_dict["train"].remove_columns(["statement", "status"])
    # Keep raw columns so it can be used for generation & accuracy checks
    test_data = dataset_dict["test"]
    
    print("✅ Dataset Prepared Successfully:")
    return train_data, test_data
    

In [11]:
train_data, test_data = prepare_dataset(dataset_path=DATASET_PATH, tokenizer=tokenizer)

Map: 100%|██████████| 10537/10537 [00:00<00:00, 12913.28 examples/s]

✅ Dataset Prepared Successfully:


## 6. Train the Model

### **For training the model**:

- **Per Sequence**: Each individual sequence (e.g., prompt + completion) cannot exceed the model's maximum token length of 32768 tokens.
- **Per Batch**: You can have multiple sequences in a batch, and their combined tokens can exceed 32768 as long as each sequence respects the 32768 token limit.

**For example**:

    If we were about to train the following two sequences with this specific model (32768 maximum tokens):
    Sequence 1: 18000 tokens and Sequence 2: 16000 tokens.

    **This would be allowed since both sequences are within the limit of 32768 tokens individually.**

    For a batch of two sequences --> Total tokens = 18000 + 16000 = 34000 tokens in the batch. 

    **This would still be allowed since batch size is irrelevant as long as individual sequences are within the limit.**

### **For inference**:
In our case if the conversation with the model exceeds the 32768 tokens, older tokens are typically removed using a sliding window approach to make room for new tokens. That means that the model loses context from the beginning of the conversation.

In [12]:
def clear_gpu_memory():
    """Forces the GPU to release 'reserved' memory back to the system."""
    torch.cuda.empty_cache()
    gc.collect()
    print("🧹 GPU Memory Cleared.")

In [ ]:
def train_model(model, tokenizer: AutoTokenizer, lora_mode: str, train_data = train_data, test_data = test_data):
    """Configures and runs the SFT Training pipeline."""
    
    print(f"🚀 Starting training with [{lora_mode.upper().replace('O', 'o')}]")
    
    # 1. Clean gpu memory
    clear_gpu_memory()
    
    # 2. Configure trainer
    training_args = SFTConfig(
        # Training parameters
        dataset_text_field="text",                  # Specifies the column name in the dataset containing the input text
        output_dir = os.path.dirname(ADAPTER_DIR),  # Directory where model checkpoints and logs will be saved
        max_steps = 1200,                           # Total number of training steps to perform
        per_device_train_batch_size = 2,            # Batch size per device during training
        per_device_eval_batch_size = 2,             # Batch size per device during evaluation
        learning_rate = 2e-4,                       # Initial learning rate for the optimizer.
        optim = "adamw_8bit",                       # Optimizer to use (8-bit version of AdamW, uses less memory and is faster)
        weight_decay = 0.01,                        # Adds L2 regularization to prevent overfitting
        lr_scheduler_type = "linear",               # Learning rate will decay linearly from the initial value to 0
        max_length = 1024,                          # Maximum number of tokens the model will see in each input
        warmup_steps = 200,                         # Slowly increase learning rate from 0 to the target value in the first 200 steps (helps stabilize early training)               
        seed = 42,                                  # Seed for reproducibility of training
        
        # Precision & Memory Optimization
        bf16 = True,                                # bfloat16 precision for faster training and reduced memory usage (recommended for RTX 40 series)
        bf16_full_eval = True,                      # Evaluate in bfloat16 to avoid HybridCache .float() bug (needed if Transformers < 4.50.1 or Accelerate < 1.4.0; safe to remove after upgrade)
        fp16 = False,                               # float16 precision for older GPUs (e.g., RTX 30 series, T4). If both bf16 and fp16 are set to True, Trainer will prioritize bf16, as the two cannot be used together.
        gradient_checkpointing = True,              # Saves GPU memory by not storing forward pass data. It recomputes them during backpropagation (trades speed for memory)
        gradient_accumulation_steps = 8,            # Combine gradients from 1 different batch before updating weights (simulates larger batch size with less memory)
        gradient_checkpointing_kwargs={"use_reentrant": False},
        
        # Evaluation & Logging
        logging_steps = 10,                         # Frequency (in steps) to log training metrics.
        eval_strategy = "steps",                    # Evaluation strategy to adopt during training
        eval_steps = 600,                           # Frequency (in steps) to run evaluation.
        save_steps = 600,                           # Strategy to save checkpoints
        save_total_limit = 1,                       # Keep only the best checkpoint
        load_best_model_at_end = True,              # Restores best checkpoint after training
        metric_for_best_model = "eval_loss",        # Use lowest validation loss
        greater_is_better = False,                  # Because lower loss = better
    )

    # 3. Initialize trainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=test_data,
        processing_class=tokenizer,                  
    )

    # Quick check before training starts
    # This grabs the first processed example from the trainer's processed dataset
    processed_sample = trainer.train_dataset[0]
    print(f"Length of first sample: {len(processed_sample['input_ids'])}")
    print(f"Max Length Allowed: {training_args.max_length}")
    # If Length == Max Length, it was likely truncated.

    # 4. Train
    start_gpu_time = timer()
    trainer.train()
    end_gpu_time = timer()
    
    print(f"🏁 Training complete. Time: {(end_gpu_time - start_gpu_time) / 60:.2f} minutes")

    # 5. Save only the adapters
    print(f"💾 Saving adapters to: {ADAPTER_DIR}...")
    trainer.model.save_pretrained(ADAPTER_DIR)
    trainer.processing_class.save_pretrained(ADAPTER_DIR) 
    
    del model
    del trainer
    clear_gpu_memory()

In [20]:
train_model(model=model, tokenizer=tokenizer, lora_mode=LORA_MODE, train_data=train_data, test_data=test_data)

🚀 Starting training with [QLoRA]
🧹 GPU Memory Cleared.


Truncating eval dataset: 100%|██████████| 10537/10537 [00:00<00:00, 778594.88 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


--- INSIDE SFT TRAINER DATA ---
Length of first sample: 118
Max Length Allowed: 1024


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 5. Load the adapters on top of the base model

Because Unsloth saves an `adapter_config.json` file along with the model adapters, which records the original base model name or path used for training, `FastModel.from_pretrained()` automatically reads this file, loads the correct base model, and attaches the fine-tuned adapters without needing to manually specify the base model again.

In [32]:
base_model = AutoModelForCausalLM.from_pretrained(gemma_3, attn_implementation="eager").to(device)

In [38]:
from peft import PeftConfig, PeftModel

adapter_dir = "./sft_output/gemma-3-adapters"

loaded_model = PeftModel.from_pretrained(base_model, adapter_dir)
loaded_tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

In [39]:
from transformers import TextStreamer

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Hello how are you?.",}]
}]

text = loaded_tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, 
)

outputs = loaded_model.generate(
    **loaded_tokenizer(text, add_special_tokens = False, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048, 
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(loaded_tokenizer, skip_prompt = True),
)

Normal<end_of_turn>


In [40]:
loaded_tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nHello how are you?.<end_of_turn>\n<start_of_turn>model\nNormal<end_of_turn>']

## 6. Apply Prompt Engineering

In [41]:
df_train["status"].unique()

array(['Anxiety', 'Normal', 'Depression', 'Suicidal', 'Stress', 'Bipolar',
       'Personality disorder'], dtype=object)

In [42]:
# Prompt with instructions
prompt = [
    #    ----------- System -----------
    {
        "role": "system",
        "content":
            """You are an AI assistant that offers *non-clinical* emotional support. Your goals are: 
1. Classify every user message into one of seven mental-health labels (Normal, Depression, Suicidal, Anxiety, Stress, Bi-Polar, Personality Disorder) 
2. Produce an empathic, concise reply that fits your role using just one of the labels.
3. Never give medical or legal advice; encourage professional help instead."""
    },

    #    ----------- Example -----------
    {
        "role": "user",
        "content": "I failed my exam and I feel useless.",
    },
    {
        "role": "assistant",
        "content":
            "Depression. Get advice from a professional"
    },

    #  ----------- User request -----------
    {
        # Normal
        # "role": "user",
        # "content": "Today was a pretty regular day. Nothing special, but I’m feeling okay overall.",

        # Depression
        # "role": "user",
        # "content": "I feel empty and disconnected from everything. It’s like I’m just going through the motions.",
        
        # Suicidal
        # "role": "user",
        # "content": "I feel like there's no point in going on anymore. Everything hurts and I just want the pain to stop.",

        # Anxiety
        # "role": "user",
        # "content": "My mind won't stop racing about everything that could go wrong. I can't relax no matter what I do.",

        # Stress
        "role": "user",
        "content": "I’m overwhelmed with work and personal problems. I can’t seem to catch a break, and it’s exhausting.",

        # Bi-Polar
        # "role": "user",
        # "content": "This morning I felt unstoppable, like I could do anything. Now I can barely get out of bed.",

        # Personality Disorder
        # "role": "user",
        # "content": "I don't know who I am anymore. I change depending on who I'm with, and it’s exhausting.",
    },
]

In [43]:
# Format the prompt using the chat template
formated_prompt = loaded_tokenizer.apply_chat_template(
    prompt,
    tokenize = False,
    add_generation_prompt = True
)

# Tokenize the formatted prompt
tokenized_prompt = loaded_tokenizer(formated_prompt, return_tensors="pt", add_special_tokens = False).to("cuda")

# Print number of input tokens
num_tokens = tokenized_prompt.input_ids.shape[1]
print(f"Number of input tokens: {num_tokens}")

Number of input tokens: 155


In [44]:
# Generate a model response based on the tokenized prompt
model_output = loaded_model.generate(
    **tokenized_prompt,
    max_new_tokens = 2048,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
    streamer = TextStreamer(loaded_tokenizer, skip_prompt = True)
)

Stress<end_of_turn>


---